## 1. Importing the libraries

| Import | Purpose |
| --- | --- |
| json | Converts JSON-formatted genres and keywords into Python objects. |
| re | Cleans names by removing non-alphanumeric characters using regular expressions. |
| pickle | Saves the trained preprocessing and recommendation artifacts into a file. |
| numpy | Provides numerical operations (imported but not directly used in this code). |
| pandas | Loads and manipulates the movie dataset using DataFrames. |
| Path | Creates and handles filesystem paths conveniently. |
| TfidfVectorizer | Converts movie text into a sparse numerical TF-IDF matrix. |
| linear_kernel | Computes similarity scores efficiently - for L2-normalized TF-IDF vectors the dot product equals cosine similarity.


In [2]:
import json
import re
import pickle
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

## 2. Loading the raw dataset

- `pd.read_csv()` loads the CSV into a DataFrame: **35,001 movies, 12 columns**.
- `isnull().sum()` counts missing values per column. Notably: overview 3,342, poster_url 3,549, cast 1,658, director 1,007.
- `duplicated().sum()` checks repeats: **0 duplicate movie IDs**, but **1,665 duplicate-title rows** (the first occurrence is not marked duplicated; different films can share a name). Titles are checked but NOT removed at this stage.


In [3]:
DATA_PATH = "../data/raw/movies_raw_enriched.csv"

movies_df = pd.read_csv(DATA_PATH)

print("Raw dataset shape:", movies_df.shape)

print("\nMissing values:")
print(movies_df.isnull().sum())

print(
    "\nDuplicate movie IDs:",
    movies_df["movie_id"].duplicated().sum()
)

print(
    "Duplicate titles:",
    movies_df["title"].duplicated().sum()
)

Raw dataset shape: (35001, 12)

Missing values:
movie_id           0
title              0
overview        3342
release_date      42
genres             0
keywords           0
runtime            0
vote_average       0
vote_count         0
poster_url      3549
cast            1658
director        1007
dtype: int64

Duplicate movie IDs: 0
Duplicate titles: 1665


## 3. Selecting columns and removing missing data

- Keep only **7 columns** used by the recommender: movie_id, title, overview, genres, keywords, cast, director. Runtime, ratings, release date and poster URL are NOT used to compute similarity.
- `.copy()` creates a separate DataFrame instead of a view of the original data.
- `dropna()` removes every row containing a missing value in any selected column: **29,820 movies retained** (35,001 - 29,820 = 5,181 removed, ~85.2% kept). Because dropna runs after column selection, missing values in excluded columns (release_date, poster_url, ...) do not cause removal. Trade-off: incomplete but usable movies are lost.
- `reset_index(drop=True)` builds a fresh 0..n-1 index without keeping the old index as a column.


In [9]:
movies = movies_df[
    [
        "movie_id",
        "title",
        "overview",
        "genres",
        "keywords",
        "cast",
        "director"
    ]
].copy()

print("\nSelected dataset shape:", movies.shape)

movies = movies.dropna().reset_index(drop=True)

print("After dropping missing values:", movies.shape)
print("Total movies:", len(movies))


Selected dataset shape: (35001, 7)
After dropping missing values: (29820, 7)
Total movies: 29820


## 4. Feature weights

Weights control how many times each field is repeated in the movie's text "soup": genres 4, keywords 1, cast 3, director 3. They are **heuristic weights implemented through repetition, not learned from training**, and they influence the representation BEFORE similarity is computed (similarity scores are not multiplied by them afterwards).


In [10]:
# Feature weights: how many times each field is repeated in the text "soup"
# Recommendation based on the importance of genres, keywords, cast and director (you can vary according to your choice)
W_GENRES = 4
W_KEYWORDS = 1
W_CAST = 3
W_DIRECTOR = 3

## 5. Converting JSON metadata into Python lists

`convert()` parses each CSV cell like `'["Action", "Drama"]'` into a real Python list with `json.loads()`.
- Returns the parsed list if it is a list; otherwise `[]`.
- Catches `JSONDecodeError` (invalid JSON) and `TypeError` (unsuitable input type) so one bad cell never stops the pipeline.
- Needed because the CSV stores genres/keywords as JSON strings; parsing lets us process each value individually and repeat them by weight.


In [11]:
def convert(obj):
    try:
        result = json.loads(obj)

        if isinstance(result, list):
            return result

        return []

    except (json.JSONDecodeError, TypeError):
        return []


movies["genres"] = movies["genres"].apply(convert)
movies["keywords"] = movies["keywords"].apply(convert)

## 6. Normalizing names with `squash()`

`re.sub(r"[^a-z0-9]", "", name.lower())` lowercases the name, then removes every character that is not a lowercase letter or digit.
- "Spider-Man" -> "spiderman", "Robert Downey Jr." -> "robertdowneyjr".
- Creates a consistent representation, reducing differences from spaces, hyphens and punctuation.
- Applied only to genres, keywords, cast and director - not to the overview or the title used for searching.
- Limitation: strips non-ASCII letters, so accented/non-English names lose information.


In [12]:
def squash(name):
    return re.sub(r"[^a-z0-9]", "", name.lower())

## 7. Building the movie text - the "soup"

Combines all content features into one text representation per movie:
- **genres / keywords**: list comprehensions keep valid non-empty strings, normalized with `squash()`.
- **cast**: `row["cast"].split("|")[:5]` limits to the **first 5** cast members (in existing order, not verified prominence).
- **director**: split on `|`, no limit.
- The `" ".join([overview.lower()] + genres*W + ...)` concatenation implements the weights via list repetition.
- The soup captures similarity beyond the plot alone: shared genres, themes, actors and directors. Repetition influences TF-IDF but does not guarantee a feature contributes exactly 4x to the similarity score (TF-IDF weighting and vector normalization also matter).


In [13]:
def build_text(row):

    genres = [
        squash(g)
        for g in row["genres"]
        if isinstance(g, str) and g.strip()
    ]

    keywords = [
        squash(k)
        for k in row["keywords"]
        if isinstance(k, str) and k.strip()
    ]

    cast = [
        squash(c)
        for c in row["cast"].split("|")[:5]
        if c.strip()
    ]

    director = [
        squash(d)
        for d in row["director"].split("|")
        if d.strip()
    ]

    return " ".join(
        [row["overview"].lower()]
        + genres * W_GENRES
        + keywords * W_KEYWORDS
        + cast * W_CAST
        + director * W_DIRECTOR
    )


movies["soup"] = movies.apply(build_text, axis=1)

## 8. Creating the final dataset

- `movies.apply(build_text, axis=1)` builds the soup row by row into a new `soup` column.
- `new_df` keeps only **movie_id, title, soup** - genres/keywords/cast/director are no longer needed separately because they are inside the soup.
- Resetting the index guarantees row positions in `new_df` align with rows of the TF-IDF matrix built next.
- Final shape: **(29820, 3)** - no movies are removed in this step.


In [14]:
new_df = movies[
    [
        "movie_id",
        "title",
        "soup"
    ]
].copy()

new_df = new_df.reset_index(drop=True)

print("\nFinal dataset shape:", new_df.shape)
print("Total movies:", len(new_df))


Final dataset shape: (29820, 3)
Total movies: 29820


## 9. TF-IDF: converting text into numerical features

Each movie's soup is one **document**; all 29,820 soups form the corpus. TF-IDF weights a term by term frequency and inverse document frequency (rare-across-corpus terms get more weight).

Parameters:
- `stop_words="english"` - removes common English words (the, is, and) with little distinguishing power.
- `ngram_range=(1, 1)` - unigrams only (no bigrams like "science fiction").
- `min_df=2` - a term must appear in at least 2 documents to enter the vocabulary.
- `max_features=100_000` - an upper limit on vocabulary size, not a target.
- `sublinear_tf=True` - scales term frequency as 1 + log(TF), dampening repeated terms.

`fit_transform()` learns the vocabulary and IDF values (`fit`) and converts each soup into a vector (`transform`).
Result: a **sparse matrix** of **29,820 movies x 56,336 features** - sparse because each movie uses only a small slice of the full vocabulary, so only non-zero values are stored.

TF-IDF is a statistical feature-extraction technique - **not** a deep learning algorithm and does not recommend by itself; it only creates the vectors that are compared next.


In [15]:
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 1),
    min_df=2,
    max_features=100_000,
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(new_df["soup"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (29820, 56336)


## 10. The recommendation function

`recommend(title, movie_id=None, n=5)` generates top-n recommendations.

1. **Validate title** - must be a non-empty string, else return an error message.
2. **Validate n** - zero or negative returns an empty DataFrame (`iloc[0:0]` keeps the column structure with 0 rows).
3. **Normalize input**: `strip().casefold()`; the same normalization is applied to every dataset title so matching is case-insensitive.
4. **Exact matches** - boolean mask `normalized_titles == key` collects matching `new_df.index` values as a list.
5. **No exact match** - substring search over unique titles (`if key in t.strip().casefold()`), up to 5 suggestions returned in the message. This is substring matching, NOT fuzzy matching, so it will not correct spelling mistakes.
6. **Duplicate titles** - if a `movie_id` is supplied, matches are filtered against it (both values compared as strings so int/string IDs match); otherwise the first exact title match is used.

## 11. Calculating similarity between movies

- `movie_index = matches[0]` selects the first matched movie's row position; rows of `new_df` align with rows of the TF-IDF matrix.
- `linear_kernel(tfidf_matrix[movie_index], tfidf_matrix)` computes the **dot product of the selected movie's vector with every movie's vector**; `.ravel()` flattens the scores into 1-D.

Because `TfidfVectorizer` L2-normalizes vectors by default, the dot product **equals cosine similarity** - the cosine of the angle between two vectors (higher = more similar; scores in [0, 1] for non-negative TF-IDF vectors). This is why `linear_kernel` is used instead of `cosine_similarity`: same result, computed efficiently.

Only the selected movie vs. all others is computed (29,820 scores) instead of a full 29,820^2 = 889,232,400-entry similarity matrix.

## 12. Ranking and selecting recommendations

- Create a `movie_id`/`title` DataFrame and attach each movie's similarity score.
- **Exclude** the selected row itself (`candidates.index != movie_index`) AND every record with the same normalized title (`normalized_titles != selected_title`); both conditions must hold, so the movie is never recommended to itself.
- `sort_values("similarity", ascending=False, kind="stable").head(n)` - highest scores first; `kind="stable"` preserves order for equal scores.
- Return only `movie_id` and `title` with a fresh index; similarity is used internally for ranking and is not returned.


In [16]:
def recommend(title, movie_id=None, n=5):

    if not isinstance(title, str) or not title.strip():
        return "Please provide a valid movie title."

    if n <= 0:
        return new_df.iloc[0:0][
            ["movie_id", "title"]
        ].copy()

    key = title.strip().casefold()

    normalized_titles = (
        new_df["title"]
        .str.strip()
        .str.casefold()
    )

    matches = new_df.index[
        normalized_titles == key
    ].tolist()

    if not matches:

        close = [
            t
            for t in new_df["title"].dropna().unique()
            if key in t.strip().casefold()
        ][:5]

        return (
            f"'{title}' not found. "
            f"Similar titles: {close}"
        )

    if movie_id is not None:

        matches = [
            i
            for i in matches
            if str(new_df.iloc[i]["movie_id"])
            == str(movie_id)
        ]

        if not matches:
            return (
                f"Movie ID {movie_id} "
                f"does not match '{title}'."
            )

    movie_index = matches[0]

    selected_title = normalized_titles.iloc[movie_index]

    similarities = linear_kernel(
        tfidf_matrix[movie_index],
        tfidf_matrix
    ).ravel()

    candidates = new_df[
        ["movie_id", "title"]
    ].copy()

    candidates["similarity"] = similarities

    candidates = candidates[
        (candidates.index != movie_index)
        & (normalized_titles != selected_title)
    ].copy()

    results = candidates.sort_values(
        "similarity",
        ascending=False,
        kind="stable"
    ).head(n)

    return results[
        ["movie_id", "title"]
    ].reset_index(drop=True)

## 13. Testing the recommendation system

A manually selected list of titles spanning action, sci-fi, romance, Indian cinema, animation, crime, horror and fantasy ("3 Idiots" is intentionally tested twice). For each title:
- Call `recommend(movie)`.
- If the result is a `str`, it is an error / "not found + similar titles" message; otherwise print the recommendation table without the index.

This is a **basic functional test**: it verifies the function runs, handles missing titles and produces lists. It does **not** compute formal evaluation metrics (Precision@K, Recall@K, NDCG@K).


In [17]:
test_movies = [

    "The Dark Knight",
    "Iron Man",
    "Spider-Man",

    "Interstellar",
    "Inception",
    "The Matrix",

    "The Notebook",
    "Titanic",
    "La La Land",

    "3 Idiots",
    "Home Alone",
    "The Hangover",

    "Toy Story",
    "Finding Nemo",
    "Coco",

    "The Godfather",
    "Se7en",
    "Gone Girl",

    "The Conjuring",
    "A Nightmare on Elm Street",

    "Drishyam",
    "Dangal",
    "3 Idiots",

    "The Lord of the Rings",
    "Harry Potter and the Philosopher's Stone",

    "Shutter Island",
    "The Prestige"
]

In [18]:
for movie in test_movies:

    print(f"\n{'=' * 60}")
    print(f"Recommendations for: {movie}")
    print("=" * 60)

    result = recommend(movie)

    if isinstance(result, str):
        print(result)
    else:
        print(result.to_string(index=False))


Recommendations for: The Dark Knight
 movie_id                 title
      272         Batman Begins
    49026 The Dark Knight Rises
     1124          The Prestige
      268                Batman
   414906            The Batman

Recommendations for: Iron Man
 movie_id                     title
    10138                Iron Man 2
    68721                Iron Man 3
   315635    Spider-Man: Homecoming
    99861   Avengers: Age of Ultron
   429617 Spider-Man: Far From Home

Recommendations for: Spider-Man
 movie_id                                       title
      559                                Spider-Man 3
   969681                   Spider-Man: Brand New Day
     1930                      The Amazing Spider-Man
   315635                      Spider-Man: Homecoming
   453395 Doctor Strange in the Multiverse of Madness

Recommendations for: Interstellar
 movie_id                 title
   286217           The Martian
  1368337           The Odyssey
     1124          The Prestige
  1

## 14. Saving the model artifacts

- `MODEL_DIR.mkdir(parents=True, exist_ok=True)` creates the backend directory if it does not exist.
- Artifacts dictionary:
  | Key | Purpose |
  | --- | --- |
  | movies | movie IDs, titles and soup text |
  | tfidf_vectorizer | fitted vectorizer, vocabulary and IDF values |
  | tfidf_matrix | numerical vectors for all movies |
  | version | artifact version label ("v2") |

- The backend can reuse this prepared representation instead of rebuilding it at every start. **The movies DataFrame and tfidf_matrix must stay aligned by row index**, otherwise a similarity score could be matched to the wrong movie.
- `open(model_path, "wb")` opens binary write mode; `with` closes the file safely; `pickle.dump(..., protocol=pickle.HIGHEST_PROTOCOL)` serializes the dictionary to `movie_recommender.pkl` (~30.21 MB). Pickle is a **serialization mechanism, not an ML algorithm** - and pickle files should only be loaded from trusted sources, since loading an untrusted pickle can execute arbitrary code.


In [19]:
MODEL_DIR = Path("../backend")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_artifacts = {
    "movies": new_df.reset_index(drop=True),
    "tfidf_vectorizer": tfidf,
    "tfidf_matrix": tfidf_matrix,
    "version": "v2"
}

model_path = MODEL_DIR / "movie_recommender.pkl"

with open(model_path, "wb") as file:
    pickle.dump(
        model_artifacts,
        file,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print(f"\nV2 model saved to: {model_path.resolve()}")
print(f"Movies: {len(new_df):,}")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(
    f"File size: "
    f"{model_path.stat().st_size / (1024**2):.2f} MB"
)


V2 model saved to: D:\Seenema Final\Seenema\backend\movie_recommender.pkl
Movies: 29,820
TF-IDF matrix shape: (29820, 56336)
File size: 30.21 MB
